# Jvalyx Model Training Skeleton

This notebook is the starting point for the five-class event triage model and independent anomaly detector. The data team can provide the source file later; only the loader and column mapping should need adjustment.

Planned outputs:
- Classifier probabilities for classes 1 through 5
- Normalized anomaly score from an Isolation Forest
- Saved model artifacts with explicit versions

Do not use this notebook for production inference until the validation checks pass on a held-out dataset.

In [ ]:
# Run once in the notebook environment if dependencies are missing.
%pip install pandas numpy scikit-learn joblib matplotlib seaborn catboost

In [ ]:
from pathlib import Path
import json
import joblib
import sys
import numpy as np
import pandas as pd
from sklearn.ensemble import IsolationForest
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "backend").is_dir():
    PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from backend.pipeline import export_predictions_geojson

try:
    from catboost import CatBoostClassifier
except ImportError as error:
    raise ImportError("Install catboost before training the classifier.") from error

RANDOM_STATE = 42
MODEL_VERSION = "catboost-isoforest-0.1.0"
ARTIFACT_DIR = Path("../artifacts")
ARTIFACT_DIR.mkdir(exist_ok=True)

CLASS_NAMES = {
    1: "industrial_fire_explosion",
    2: "wildfire",
    3: "mining_coal_seam_fire",
    4: "agricultural_burning",
    5: "persistent_flare_routine_heat",
}

## 1. Load the data

Expected minimum columns are listed below. The label is required for classifier training and should be supplied by the data owner. Keep raw source fields in the dataset for auditability.

In [ ]:
DATA_PATH = Path("../data/model_training.csv")
LABEL_COLUMN = "class_id"
LATITUDE_COLUMN = "latitude"
LONGITUDE_COLUMN = "longitude"
REQUIRED_COLUMNS = {
    LABEL_COLUMN,
    LATITUDE_COLUMN,
    LONGITUDE_COLUMN,
    "bright_ti4_k",
    "bright_ti5_k",
    "frp_mw",
    "scan_km",
    "track_km",
    "persistence_score",
    "facility_frp_zscore",
    "cluster_pixel_count",
    "sensor_count",
    "data_quality_score",
}

def load_training_data(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(
            f"Training data not found at {path}. Ask the data owner for the agreed CSV or Parquet export."
        )
    frame = pd.read_csv(path) if path.suffix.lower() == ".csv" else pd.read_parquet(path)
    missing = REQUIRED_COLUMNS.difference(frame.columns)
    if missing:
        raise ValueError(f"Training data is missing columns: {sorted(missing)}")
    frame[LATITUDE_COLUMN] = pd.to_numeric(frame[LATITUDE_COLUMN], errors="raise")
    frame[LONGITUDE_COLUMN] = pd.to_numeric(frame[LONGITUDE_COLUMN], errors="raise")
    if not frame[LATITUDE_COLUMN].between(-90, 90).all() or not frame[LONGITUDE_COLUMN].between(-180, 180).all():
        raise ValueError("Latitude must be between -90 and 90; longitude must be between -180 and 180.")
    if frame[LABEL_COLUMN].isin(CLASS_NAMES).all():
        frame[LABEL_COLUMN] = frame[LABEL_COLUMN].map({name: class_id for class_id, name in CLASS_NAMES.items()})
    frame[LABEL_COLUMN] = frame[LABEL_COLUMN].astype(int)
    invalid_labels = set(frame[LABEL_COLUMN].dropna()) - set(CLASS_NAMES)
    if invalid_labels:
        raise ValueError(f"Labels must be integers 1 through 5, got: {sorted(invalid_labels)}")
    return frame

# Uncomment when the data file is available.
# raw_data = load_training_data(DATA_PATH)
# raw_data.head()

## 2. Derive deterministic features

These features are calculated before ML. The facility baseline must be frozen before an anomaly window to avoid future-data leakage.

In [ ]:
FEATURE_COLUMNS = [
    "bright_ti4_k",
    "bright_ti5_k",
    "temp_ratio",
    "frp_mw",
    "scan_km",
    "track_km",
    "persistence_score",
    "facility_frp_zscore",
    "cluster_pixel_count",
    "sensor_count",
    "data_quality_score",
]

def derive_features(frame: pd.DataFrame) -> pd.DataFrame:
    features = frame.copy()
    features["bright_ti4_k"] = features["bright_ti4_k"].fillna(0.0)
    features["bright_ti5_k"] = features["bright_ti5_k"].fillna(1.0)
    features["temp_ratio"] = features["bright_ti4_k"] / features["bright_ti5_k"].clip(lower=1.0)
    numeric_columns = [column for column in FEATURE_COLUMNS if column in features]
    features[numeric_columns] = features[numeric_columns].replace([np.inf, -np.inf], np.nan).fillna(0.0)
    return features[FEATURE_COLUMNS]

# Example after data delivery:
# X = derive_features(raw_data)
# y = raw_data[LABEL_COLUMN]

## 3. Train the classifier and anomaly detector

Use a stratified holdout for the classifier. The anomaly detector is trained on normal/reference events only when that subset is available.

In [ ]:
def train_models(raw_data: pd.DataFrame) -> tuple[CatBoostClassifier, IsolationForest, pd.DataFrame, pd.DataFrame]:
    X = derive_features(raw_data)
    y = raw_data[LABEL_COLUMN]
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
    )

    classifier = CatBoostClassifier(
        loss_function="MultiClass",
        iterations=500,
        depth=6,
        learning_rate=0.05,
        random_seed=RANDOM_STATE,
        verbose=False,
    )
    classifier.fit(X_train, y_train)

    reference_events = X_train[y_train.isin([4, 5])]
    anomaly_detector = IsolationForest(
        n_estimators=200, contamination="auto", random_state=RANDOM_STATE
    )
    anomaly_detector.fit(reference_events if len(reference_events) >= 2 else X_train)
    return classifier, anomaly_detector, X_test, y_test

# classifier, anomaly_detector, X_test, y_test = train_models(raw_data)

In [ ]:
def predict_contract(
    classifier,
    anomaly_detector,
    features: pd.DataFrame,
    source_data: pd.DataFrame | None = None,
) -> pd.DataFrame:
    probabilities = classifier.predict_proba(features)
    predicted_classes = classifier.classes_[np.argmax(probabilities, axis=1)]
    anomaly_raw = -anomaly_detector.decision_function(features)
    anomaly_score = (anomaly_raw - anomaly_raw.min()) / max(anomaly_raw.max() - anomaly_raw.min(), 1e-9)
    output = pd.DataFrame({
        "class_id": predicted_classes.astype(int),
        "anomaly_score": anomaly_score.clip(0.0, 1.0),
    }, index=features.index)
    if source_data is not None:
        missing_geo = {LATITUDE_COLUMN, LONGITUDE_COLUMN}.difference(source_data.columns)
        if missing_geo:
            raise ValueError(f"Source data is missing geographic columns: {sorted(missing_geo)}")
        output[LATITUDE_COLUMN] = source_data.loc[features.index, LATITUDE_COLUMN]
        output[LONGITUDE_COLUMN] = source_data.loc[features.index, LONGITUDE_COLUMN]
    for index, class_id in enumerate(classifier.classes_):
        output[f"class_{int(class_id)}_probability"] = probabilities[:, index]
    return output

def validate_probability_contract(predictions: pd.DataFrame) -> None:
    probability_columns = [f"class_{class_id}_probability" for class_id in CLASS_NAMES]
    if set(probability_columns).difference(predictions.columns):
        raise ValueError("Predictions must include probabilities for all five classes.")
    if not np.allclose(predictions[probability_columns].sum(axis=1), 1.0, atol=1e-6):
        raise ValueError("Class probabilities must sum to 1.")

# predictions = predict_contract(classifier, anomaly_detector, X_test, raw_data)
# validate_probability_contract(predictions)
# predictions.head()

## 4. Evaluate and save artifacts

Do not promote a model based only on accuracy. Review per-class recall, confusion between industrial heat and industrial escalation, calibration, and anomaly behavior on the replay scenarios.

In [ ]:
# print(classification_report(y_test, classifier.predict(X_test), target_names=[CLASS_NAMES[i] for i in sorted(CLASS_NAMES)]))
# print(confusion_matrix(y_test, classifier.predict(X_test), labels=sorted(CLASS_NAMES)))

def save_artifacts(classifier, anomaly_detector) -> None:
    joblib.dump(classifier, ARTIFACT_DIR / f"classifier-{MODEL_VERSION}.joblib")
    joblib.dump(anomaly_detector, ARTIFACT_DIR / f"anomaly-detector-{MODEL_VERSION}.joblib")
    metadata = {
        "model_version": MODEL_VERSION,
        "class_names": CLASS_NAMES,
        "feature_columns": FEATURE_COLUMNS,
        "label_column": LABEL_COLUMN,
    }
    (ARTIFACT_DIR / f"metadata-{MODEL_VERSION}.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")

# save_artifacts(classifier, anomaly_detector)

## 5. Export predictions as GIS features

Predictions retain the source event coordinates and are exported as a GeoJSON `FeatureCollection`. GeoJSON uses `[longitude, latitude]` coordinate order, so the resulting file can be loaded by Leaflet, QGIS, or other GIS tools.

In [ ]:
GIS_OUTPUT_PATH = Path("../artifacts/predictions.geojson")

# Run after training and evaluation:
# predictions = predict_contract(classifier, anomaly_detector, X_test, raw_data)
# validate_probability_contract(predictions)
# exported_path = export_predictions_geojson(
#     predictions.to_dict(orient="records"),
#     GIS_OUTPUT_PATH,
#     latitude_column=LATITUDE_COLUMN,
#     longitude_column=LONGITUDE_COLUMN,
# )
# print(f"Exported {len(predictions)} GIS features to {exported_path}")